# CrocoDashChecks

Checks that CrocoDash works end to end on NCAR's systems. It builds CrocoDash's grids and
forcings on every grid in its [test matrix](#matrix), from seams to poles; every week it builds
real CESM cases with CrocoDash and runs them on Derecho; every day it checks that every data
source CrocoDash reads still works. All of them test the latest CrocoDash `main`. This page
shows where each check stands.

- [How it works](how_it_works.md): what each check does, when it runs, and how to run it
  yourself.
- [Performance](performance.ipynb): how long CrocoDash's expensive steps take, with fits for
  sizing jobs.

```{toctree}
:hidden:
performance
how_it_works
integration
```

In [ ]:
import checks
from checks import FAIL, NOT_RUN, OK, WARN

runs = checks.load(checks.CESM_RUNS)  # the weekly CESM runs
health = checks.load(checks.HEALTH)  # the daily data-access check
wf = checks.workflow()  # the workflow on the matrix grids, run by hand
grids = checks.matrix()

In [ ]:
def summary_row(name, results, counts, word, kind):
    """The check, its result, when it last ran, and the CrocoDash it tested."""
    if results is None:
        return [name, f"{WARN} No results yet", "", ""]
    ok, total = counts(results)
    if ok == total:
        result = f"{OK} {ok}/{total} {word}"
    else:
        result = f"{FAIL} {ok}/{total} {word}: {total - ok} failing"
    if checks.is_stale(results["date"], kind):
        days = checks.age_days(results["date"])
        result += f" ({WARN} stale: {days} days old, so its schedule may have stopped)"
    crocodash = checks.commit(results.get("crocodash"), "CrocoDash")
    return [name, result, checks.when(results["date"]), crocodash]


def workflow_row():
    """The same for the workflow, which runs by hand, so it is never stale."""
    name = "[Workflow on the matrix grids](#matrix)"
    if wf is None:
        return [name, f"{WARN} No results yet", "", ""]
    marks = [mark for mark, _ in wf["outcomes"].values()]
    ok, failed, skipped = marks.count(OK), marks.count(FAIL), marks.count(NOT_RUN)
    result = f"{OK if ok == len(marks) else FAIL} {ok}/{len(marks)} steps passing"
    if failed or skipped:
        result += f": {failed} failing, {skipped} skipped"
    return [
        name,
        result,
        checks.when(wf["date"]) + " (by hand)",
        checks.commit(wf["crocodash"], "CrocoDash"),
    ]


checks.show(
    checks.table(
        ["Check", "Result", "Last run", "CrocoDash tested"],
        [
            workflow_row(),
            summary_row(
                "[Weekly CESM runs](#cesm)",
                runs,
                checks.cesm_counts,
                "tests passing",
                "cesm",
            ),
            summary_row(
                "[Data access](#data-access)",
                health,
                checks.health_counts,
                "access methods OK",
                "health",
            ),
        ],
    )
)

(matrix)=
## Test matrix

Every grid CrocoDash is tested on is a row of
[`matrix.yaml`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/integration/matrix.yaml),
which also turns each forcing and the CESM runs on or off for it. Each grid is built once, from
the [case config](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/integration/case.yaml),
and every step turned on for it reuses it ([How it works](how_it_works.md)). Each question below is
answered over the whole matrix; every grid's own results are folded underneath.

In [ ]:
OFF = "—"
# How a step that didn't pass is described.
DIDNT = {FAIL: "failed", NOT_RUN: "skipped", WARN: "has no result yet"}


def with_forcing(flag):
    """The grids a forcing is turned on for."""
    return [name for name, grid in grids.items() if flag in grid.get("forcings", [])]


def answer(test_ids, noun):
    """'✅ All 11 grids', or how many passed and why each other one didn't."""
    if wf is None:
        return f"{WARN} No results yet"
    outcomes = {t: wf["outcomes"].get(t, (WARN, "")) for t in test_ids}
    bad = {t: outcome for t, outcome in outcomes.items() if outcome[0] != OK}
    if not bad:
        return f"{OK} All {len(outcomes)} {noun}"
    whys = []
    for test_id, (mark, reason) in bad.items():
        if mark == NOT_RUN:
            reason = reason.split(":")[
                0
            ]  # a skip says why, then repeats its grid's error
        name = test_id.split("[")[1].rstrip("]")
        whys.append(
            f"`{name}` {DIDNT[mark]}" + (f": {checks.plain(reason)}" if reason else "")
        )
    mark = FAIL if FAIL in [m for m, _ in bad.values()] else NOT_RUN
    return f"{mark} {len(outcomes) - len(bad)} of {len(outcomes)} {noun}. " + "; ".join(
        whys
    )


def cesm_answer():
    """How many CESM tests on the matrix's grids passed."""
    cesm_grids = [name for name, grid in grids.items() if grid.get("cesm_runs")]
    tests = [t for t in (runs or {}).get("tests", []) if t.get("grid") in cesm_grids]
    if not tests:
        return f"{WARN} No results yet"
    ok = sum(t["passed"] for t in tests)
    if ok == len(tests):
        return f"{OK} All {ok} tests"
    return f"{FAIL} {ok} of {len(tests)} tests ([which failed](#cesm))"


sweeps = [t for t in (wf or {"outcomes": {}})["outcomes"] if "_sweep[" in t]
text = checks.table(
    ["Question", "Answer"],
    [
        [
            "Does every grid build, with its bathymetry?",
            answer([f"test_grid[{name}]" for name in grids], "grids"),
        ],
        [
            "Do open boundaries run on every grid that has them?",
            answer([f"test_bc[{name}]" for name in with_forcing("bc")], "grids"),
        ],
        [
            "Does runoff mapping run on every grid that has it?",
            answer(
                [f"test_runoff[{name}]" for name in with_forcing("runoff")], "grids"
            ),
        ],
        [
            "Do MOM_interface's regional tests run in CESM on the grids marked for it?",
            cesm_answer(),
        ],
        [
            "Do the sweeps behind the [Performance](performance.ipynb) fits run?",
            answer(sweeps, "cases"),
        ],
    ],
)
if wf is not None:
    text += (
        f"\n\nGrids and forcings run {checks.when(wf['date'])} with CrocoDash "
        f"{checks.commit(wf['crocodash'], 'CrocoDash')}."
    )
checks.show(text)

In [ ]:
def cesm_cell(name, grid):
    """How many of the CESM tests run on this grid passed."""
    if not grid.get("cesm_runs"):
        return OFF
    tests = [t for t in (runs or {}).get("tests", []) if t.get("grid") == name]
    if not tests:
        return f"{WARN} No result"
    ok = sum(t["passed"] for t in tests)
    return f"{OK if ok == len(tests) else FAIL} {ok}/{len(tests)} passed"


rows = []
for name, grid in grids.items():
    forcings = grid.get("forcings", [])
    rows.append(
        [
            f"`{name}` ({checks.resolution(grid['grid'])})",
            checks.step(wf, f"test_grid[{name}]"),
            checks.step(wf, f"test_bc[{name}]") if "bc" in forcings else OFF,
            checks.step(wf, f"test_runoff[{name}]") if "runoff" in forcings else OFF,
            cesm_cell(name, grid),
            grid.get("about", ""),
        ]
    )
checks.show(
    checks.table(
        [
            "Grid",
            "Grid & bathymetry",
            "Open boundaries",
            "Runoff mapping",
            "CESM runs",
            "About",
        ],
        rows,
    )
    + f"\n\n{OK} passed, with the time it took · {FAIL} failed · {NOT_RUN} skipped · "
    f"{WARN} no result yet · {OFF} off."
)

(cesm)=
## Weekly CESM runs on Derecho

Every Monday, each regional SMS test in MOM_interface's `testlist_mom.xml` is built with
CrocoDash on each grid the [test matrix](#matrix) turns the CESM runs on for, and run in CESM on
one Derecho node. Each test passes through
CIME's phases in order, from creating the case to running the model. More:
[Integration](integration.md); the code is in
[`integration/`](https://github.com/CROCODILE-CESM/CrocoDashChecks/tree/main/integration).

In [ ]:
# CESM's stub components, left out of the components column.
STUB_COMPONENTS = {
    "SATM",
    "SLND",
    "SICE",
    "SOCN",
    "SROF",
    "SGLC",
    "SWAV",
    "SESP",
    "SIAC",
}


def components(longname):
    """A compset's active components, e.g. 'DATM (JRA) · CICE (PRES) · MOM6'."""
    active = []
    for token in longname.split("_")[1:]:
        name, *mods = token.split("%")
        mods = [m for m in mods if m != "REGIONAL"]
        if name not in STUB_COMPONENTS:
            active.append(f"{name} ({', '.join(mods)})" if mods else name)
    return " · ".join(active)


def phase_marks(test):
    """One mark per CIME phase, in order: passed, failed or not reached."""
    status = dict(test["phases"])
    marks = {"PASS": OK, "FAIL": FAIL}
    return "".join(marks.get(status.get(phase), NOT_RUN) for phase, _ in checks.PHASES)


def result(test):
    """'Pass', or the first phase that failed."""
    if test["passed"]:
        return f"{OK} Pass"
    status = dict(test["phases"])
    failed = [label for phase, label in checks.PHASES if status.get(phase) == "FAIL"]
    return f"{FAIL} Failed: {failed[0]}" if failed else f"{WARN} Incomplete"


if runs is None:
    checks.show("No results yet.")
else:
    # Failures first. Runs before the test matrix didn't record a grid.
    tests = sorted(
        runs["tests"], key=lambda t: (t["passed"], t.get("grid", ""), t["compset"])
    )
    rows = [
        [
            f"`{t['compset']}`",
            t.get("grid", "—"),
            f"`{t['test']}`",
            components(t["longname"]),
            phase_marks(t),
            result(t),
        ]
        for t in tests
    ]
    order = " → ".join(label for _, label in checks.PHASES)
    checks.show(
        checks.table(
            ["Compset", "Grid", "Test", "Components", "Phases", "Result"], rows
        )
        + f"\n\nPhases, left to right: {order}. {OK} passed, {FAIL} failed, {NOT_RUN} not reached."
        + f"\n\nRun {checks.when(runs['date'])} with CESM {checks.commit(runs.get('cesm'), 'CESM')}"
        + f" and CrocoDash {checks.commit(runs.get('crocodash'), 'CrocoDash')}."
    )

(data-access)=
## Data access, daily on GLADE

Every day, each way CrocoDash can fetch data (every access method of every product in its
`ProductRegistry`) is called for real on Casper, with GLADE mounted and our own credentials. A
method that fails says why; one still running after 15 minutes is stopped and counted as
failing. The code is in
[`data_access/`](https://github.com/CROCODILE-CESM/CrocoDashChecks/tree/main/data_access).

In [ ]:
if health is None:
    checks.show("No results yet.")
else:
    ok, total = checks.health_counts(health)
    by_name = sorted(
        health["validate_checks"], key=lambda c: (c["product"], c["method"])
    )
    failing = [c for c in by_name if not c["ok"]]
    working = [c for c in by_name if c["ok"]]
    text = (
        f"{ok} of {total} access methods working. Checked {checks.when(health['date'])}"
        f" with CrocoDash {checks.commit(health.get('crocodash'), 'CrocoDash')}."
    )
    if failing:
        text += f"\n\n**{FAIL} Failing ({len(failing)})**\n\n" + checks.table(
            ["Product", "Access method", "Why it failed", "Time"],
            [
                [
                    c["product"],
                    f"`{c['method']}`",
                    checks.plain(c.get("detail", "")),
                    checks.fmt_s(c["elapsed_s"]),
                ]
                for c in failing
            ],
        )
    text += f"\n\n**{OK} Working ({len(working)})**\n\n" + checks.table(
        ["Product", "Access method", "Time"],
        [
            [c["product"], f"`{c['method']}`", checks.fmt_s(c["elapsed_s"])]
            for c in working
        ],
    )
    text += f"\n\nRaw results: [health.json]({checks.SITE_URL}/health.json)"
    checks.show(text)

(performance-summary)=
(mom6-scaling)=
## Performance

How long CrocoDash's expensive steps take and how much memory they need, with fits for sizing
jobs: grids with GEBCO bathymetry, open boundaries, GLOFAS runoff mapping, and MOM6's speed
against its task count. See the [Performance page](performance.ipynb).